In [0]:
import pytest

In [0]:
%run /Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/src/task1_ingestion

In [0]:
%run /Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/resources/utils

In [0]:
def test_file_path_exists():
    import os
    file_paths = [
        "/Volumes/az_ci_de_dbs/ecom_dev/ecom_src/Products.csv",
        "/Volumes/az_ci_de_dbs/ecom_dev/ecom_src/Customer.xlsx",
        "/Volumes/az_ci_de_dbs/ecom_dev/ecom_src/Orders.json"
    ]
    for path in file_paths:
        assert os.path.exists(path), f"File path does not exist: {path}"

In [0]:
def test_product_schema():
    expected_cols = ["Product ID", "Category", "Sub-Category", "Product Name", "State", "Price per product"]
    actual_cols = csv_reader("/Volumes/az_ci_de_dbs/ecom_dev/ecom_src/Products.csv").columns

    assert expected_cols == actual_cols

def test_customer_schema():
    expected_cols = ["Customer ID", "Customer Name", "email", "phone", "address", "Segment", "Country", "City", "State", "Postal Code", "Region"]
    actual_cols = excel_reader("/Volumes/az_ci_de_dbs/ecom_dev/ecom_src/Customer.xlsx", "Worksheet").columns

    assert expected_cols == actual_cols

def test_order_schema():
    expected_cols = ["Customer ID", "Discount", "Order Date", "Order ID", "Price", "Product ID", "Profit", "Quantity", "Row ID", "Ship Date", "Ship Mode"]
    actual_cols = json_reader("/Volumes/az_ci_de_dbs/ecom_dev/ecom_src/Orders.json").columns

    assert expected_cols == actual_cols

In [0]:
def test_data_exists():
    product_df = csv_reader("/Volumes/az_ci_de_dbs/ecom_dev/ecom_src/Products.csv")
    customer_df = excel_reader("/Volumes/az_ci_de_dbs/ecom_dev/ecom_src/Customer.xlsx", "Worksheet")
    order_df = json_reader("/Volumes/az_ci_de_dbs/ecom_dev/ecom_src/Orders.json")

    assert product_df.count() > 0, "Products data is empty"
    assert customer_df.count() > 0, "Customer data is empty"
    assert order_df.count() > 0, "Orders data is empty"

In [0]:
def test_catalog_and_schema_exist():
    try:
        catalog_name = "az_ci_de_dbs"
        schema_name = "ecom_dev"
        spark.sql(f"USE CATALOG {catalog_name}")
        spark.sql(f"USE SCHEMA {schema_name}")
    except Exception as e:
        assert False, f"Catalog or schema does not exist: {e}"

In [0]:
def test_apply_column_alias():
    schema_alias = {
        "id": "identifier",
        "val": "value"
    }
    df = spark.createDataFrame([(1, "A"), (2, "B")], ["id", "val"])
    aliased_df = apply_column_alias(df, schema_alias)
    expected_cols = ["identifier", "value"]
    actual_cols = aliased_df.columns
    assert expected_cols == actual_cols, f"Expected columns {expected_cols}, got {actual_cols}"

In [0]:
def test_delta_writer_success():
    # Create a small DataFrame for testing
    test_df = spark.createDataFrame(
        [("A", 1), ("B", 2)],
        ["col1", "col2"]
    )
    table_name = "ecom_dev.test_delta_writer_table"
    try:
        delta_writer(test_df, table_name, mode="overwrite")
        # Verify table exists and data is written
        result = spark.sql(f"SELECT * FROM {table_name}")
        assert result.count() == 2
    finally:
        spark.sql(f"DROP TABLE IF EXISTS {table_name}")

def test_delta_writer_none_df():
    try:
        delta_writer(None, "ecom_dev.test_delta_writer_table")
    except ValueError as e:
        assert "Cannot write a None DataFrame." in str(e)

def test_delta_writer_empty_table_name():
    test_df = spark.createDataFrame([("A", 1)], ["col1", "col2"])
    try:
        delta_writer(test_df, "")
    except ValueError as e:
        assert "table_name must be a non-empty string." in str(e)

In [0]:
def test_count_file_vs_delta_table():
    # Products
    product_df = csv_reader("/Volumes/az_ci_de_dbs/ecom_dev/ecom_src/Products.csv")
    product_file_count = product_df.count()
    product_schema_alias = {
        "Product ID": "product_id",
        "Category": "category",
        "Sub-Category": "sub_category",
        "Product Name": "product_name",
        "State": "state",
        "Price per product": "price_per_product"
    }
    product_aliased_df = apply_column_alias(product_df, product_schema_alias)
    product_table = "az_ci_de_dbs.ecom_dev.stg_products"
    delta_writer(product_aliased_df, product_table, mode="overwrite")
    product_delta_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {product_table}").collect()[0]["cnt"]
    assert product_file_count == product_delta_count, f"Products file count ({product_file_count}) != Delta table count ({product_delta_count})"
    spark.sql(f"DROP TABLE IF EXISTS {product_table}")

    # Customers
    customer_df = excel_reader("/Volumes/az_ci_de_dbs/ecom_dev/ecom_src/Customer.xlsx", "Worksheet")
    customer_file_count = customer_df.count()
    customer_schema_alias = {
        "Customer ID": "customer_id",
        "Customer Name": "customer_name",
        "email": "email",
        "phone": "phone",
        "address": "address",
        "Segment": "segment",
        "Country": "country",
        "City": "city",
        "State": "state",
        "Postal Code": "postal_code",
        "Region": "region"
    }
    customer_aliased_df = apply_column_alias(customer_df, customer_schema_alias)
    customer_table = "az_ci_de_dbs.ecom_dev.stg_customers"
    delta_writer(customer_aliased_df, customer_table, mode="overwrite")
    customer_delta_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {customer_table}").collect()[0]["cnt"]
    assert customer_file_count == customer_delta_count, f"Customers file count ({customer_file_count}) != Delta table count ({customer_delta_count})"
    spark.sql(f"DROP TABLE IF EXISTS {customer_table}")

    # Orders
    order_df = json_reader("/Volumes/az_ci_de_dbs/ecom_dev/ecom_src/Orders.json")
    order_file_count = order_df.count()
    order_schema_alias = {
        "Row ID": "row_id",
        "Order ID": "order_id",
        "Order Date": "order_date",
        "Ship Date": "ship_date",
        "Ship Mode": "ship_mode",
        "Customer ID": "customer_id",
        "Product ID": "product_id",
        "Quantity": "quantity",
        "Price": "price",
        "Discount": "discount",
        "Profit": "profit"
    }
    order_aliased_df = apply_column_alias(order_df, order_schema_alias)
    order_table = "az_ci_de_dbs.ecom_dev.stg_orders"
    delta_writer(order_aliased_df, order_table, mode="overwrite")
    order_delta_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {order_table}").collect()[0]["cnt"]
    assert order_file_count == order_delta_count, f"Orders file count ({order_file_count}) != Delta table count ({order_delta_count})"
    spark.sql(f"DROP TABLE IF EXISTS {order_table}")

In [0]:
def main_ingestion_test():
    # Explicitly list test functions
    test_functions = [
        test_file_path_exists,
        test_product_schema,
        test_customer_schema,
        test_order_schema,
        test_data_exists,
        test_catalog_and_schema_exist,
        test_apply_column_alias,
        test_delta_writer_success,
        test_delta_writer_none_df,
        test_delta_writer_empty_table_name,
        test_count_file_vs_delta_table
    ]
    print(f"collected {len(test_functions)} items\n")

    passed = 0
    failed = 0

    for func in test_functions:
        name = func.__name__
        try:
            func()
            print(f"PASSED  {name}")
            passed += 1
        except AssertionError as e:
            print(f"FAILED  {name}")
            if str(e):
                print(f"        AssertionError: {e}")
            failed += 1
        except Exception as e:
            print(f"ERROR   {name}: {type(e).__name__}: {e}")
            failed += 1

    print(f"\n{'=' * 60}")
    result = "ALL PASSED" if failed == 0 else f"{failed} FAILED"
    print(f"{passed} passed, {failed} failed — {result}")